In [2]:
# Note: this script requires the program FFMPEG to be installed and accessible from the command line.
# FFMPEG can be downloaded from https://ffmpeg.org/download.html
# Timestamps in the subtitle files are written based on the date_modified of the file, its framerate (extracted from the video) and the photos/second taken for the timelapse (user-supplied)
# If the camera date-time was off, and therefore the modification date, you can use the program XNviewMP to batch-correct the timestamp (select files, use Metadata -> Change Timestamp)

# The script contains five sections:
# 1. Function definitions
# 2. Compress a single video (example)
# 3. Combine two videos into a single compressed video (example)
# 4. Run a single job using the defined function. Combine all videos from an input folder into a single compressed video with timestamp subtitles. Crop the start and end if desired.
# 5. Batch processing: run multiple jobs in parallel, from a excel list of input folders, output file names etc.

# Note: to write correct timestamp subtitles, remember to set how many photos per second were taken for the timelapse videos, using variable sub_step_real_s
 
from pathlib import Path
from datetime import datetime, timezone, timedelta
import cv2 # requires package opencv. Use e.g. conda install -c conda-forge opencv
import pandas as pd
import math, subprocess, json, warnings

In [1]:
1+2

3

In [3]:
# 1) Define functions
folder_ffmpeg = r"C:\Program Files\EibolSoft\FFmpeg Batch AV Converter"
folder_ffprobe = r'C:\Users\dpoppema\Downloads' # Probably in the same folder as ffmpeg normally, separate for me

ffmpeg_exe  = str(Path(folder_ffmpeg) / "ffmpeg.exe")
ffprobe_exe = str(Path(folder_ffprobe) / "ffprobe.exe")

In [8]:
# 2) SIMPLY ENCODE A SINGLE VIDEO, trim and burn-in subtitles
file_in = r'O:\HybridDune experiment\GoPros\S2\HybridDune timelapse - S2, storm 1, tide 0.mkv'
file_out = r'O:\HybridDune experiment\GoPros\S2\test out.mkv'

# Ensure output folder exists
Path(file_out).parent.mkdir(parents=True, exist_ok=True)


# Get video duration (to calculate trim end)
import subprocess, json
probe_cmd = [
    str(ffprobe_exe),
    '-v', 'error',
    '-select_streams', 'v:0',
    '-show_entries', 'format=duration',
    '-of', 'json',
    file_in
]
probe_result = subprocess.run(probe_cmd, capture_output=True, text=True)
video_info = json.loads(probe_result.stdout)
duration = float(video_info['format']['duration'])

# Calculate start and end times
start_trim = 15  # seconds to trim from start
end_trim = 3*60 + 45  # seconds to trim from end
out_duration = duration - start_trim - end_trim

# Build ffmpeg command: trim, re-encode, burn-in subtitles
cmd = [
    str(ffmpeg_exe),
    '-i', file_in,
    '-ss', str(start_trim),
    '-t', str(out_duration),
    '-vf', "subtitles='{}'".format(file_in.replace('\\', '/')),
    '-c:v', 'hevc_qsv',
    '-preset', 'medium',
    '-global_quality', '26',
    '-look_ahead', '1',
    '-r', '30000/1001',
    '-an',  # remove audio (optional)
    file_out
]

# Run ffmpeg
print(f"Running: {' '.join(cmd)}")
result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode == 0:
    print(f"Success! Video saved to: {file_out}")
else:
    print("Error during encoding:")
    print(result.stderr)


Running: C:\Program Files\EibolSoft\FFmpeg Batch AV Converter\ffmpeg.exe -i O:\HybridDune experiment\GoPros\S2\HybridDune timelapse - S2, storm 1, tide 0.mkv -ss 15 -t 16.255999999999972 -vf subtitles='O:/HybridDune experiment/GoPros/S2/HybridDune timelapse - S2, storm 1, tide 0.mkv' -c:v hevc_qsv -preset medium -global_quality 26 -look_ahead 1 -r 30000/1001 -an O:\HybridDune experiment\GoPros\S2\test out.mkv
Error during encoding:
ffmpeg version 8.0-full_build-www.gyan.dev Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 15.2.0 (Rev8, Built by MSYS2 project)
  configuration: --enable-gpl --enable-version3 --enable-static --disable-w32threads --disable-autodetect --enable-fontconfig --enable-iconv --enable-gnutls --enable-lcms2 --enable-libxml2 --enable-gmp --enable-bzlib --enable-lzma --enable-libsnappy --enable-zlib --enable-librist --enable-libsrt --enable-libssh --enable-libzmq --enable-avisynth --enable-libbluray --enable-libcaca --enable-libdvdnav --enable-libdvdrea